# modeling_v8.ipynb — Phase 3(피처 구현) + M0a·M0b·M-T·M1·M2·M3

> **modeling_v8 Phase 3~4** · 실행 위치: `modeling_v8/` 폴더 안 (상대경로 `../문제1(하)`). 커널 **venv (Python 3.12)**. PLAN **v1.5** 반영.
> 원본 `pm_feature/`를 **이식**해 피처 테이블을 만들고(Phase 3), 복원 pkl로 **M0a**(정합, valid≈38.3) → **M0b**(재학습, G1 완결) → **M-T**(시간-only 대조) → **M1**(+C23_te) → **M2**(센서 풀 gain 재선별)까지 수행한다.

## 레짐 피처 (v1.5 — 조용/요란 verdict)
`pm_log.json`은 **verdict 포맷** `[{"date","type","verdict"}]`을 쓴다. **`is_high_regime`**은 최근 PM의 `type`이 아니라 **verdict 상태기계**로 결정한다 — `loud`면 상태 갱신, `quiet`면 유지. **`high_regime_days`**의 앵커는 **"마지막 loud 이벤트"**다(임의 PM 아님) — quiet PM에서 dslp만 리셋되고 hrd는 연속돼 *가짜 신품 스파이크*를 막는다.
> **동치**: 데이터 내 유일 경계(12-24)가 loud이자 major라, verdict 규칙과 옛 type 규칙이 **현 데이터에서 값 동일**(Cell 4 assert). 그래서 M0a~M-T 수치는 전부 불변. 원본 pkl은 옛 이름 `is_post_pm`/`post_pm_days`로 학습됐고 현 데이터에선 `is_high_regime`/`high_regime_days`와 값 동일 → M0a는 옛 이름으로 pkl에 먹인다.

## 게이트
- **G1(a)**: M0a valid ≈ 38.3(±1.5) → 피처 테이블이 원본과 정합함을 증명.
- **G1**: M0b(재학습) valid ≤ 42 & CV ~41 → v8 파이프라인의 CV/valid 확정.
- **M1 채택**: ΔCV(M1−M0b) ≤ −0.3 → C23_te 채택, 아니면 기각.


In [1]:
import sys; print(sys.executable)

C:\Users\Dell3571\Documents\defect_test_prediction_MK\venv\Scripts\python.exe


In [2]:
# ── [Cell 1] 설정 · 상수(config 이식) · pm_log 파서 · pkl 로드 ───────
import json, pickle, warnings
from pathlib import Path
import numpy as np, pandas as pd, lightgbm as lgb
warnings.filterwarnings("ignore")

BASE   = Path("..")
DATA   = BASE / "문제1(하)"
ANS    = BASE / "문제1_하_answer"
PM_LOG = BASE / "pm_log.json"
PKL    = BASE / "pm_feature" / "lgbm_model.pkl"

# 원본 config.py 이식
COLS_ALL_NULL = ['C2','C13','C26','C37','C43','C47','C53','C55']
COLS_CONSTANT = ['C3','C8','C14','C19','C21','C24','C28','C29','C30','C44','C45','C51']
COLS_DUPLICATE= ['C36','C35','C38']
COLS_TO_DROP  = COLS_ALL_NULL + COLS_CONSTANT + COLS_DUPLICATE                 # 23개
FDC_FEATURES  = ['C4','C6','C12','C17','C18','C25','C27','C32','C42','C46','C48','C49',
                 'C50','C52','C54','C56','C57','C58','C59','C60','C61','C62','C63']    # 23종
AGG_FUNCS = ['mean','std','max','min','last']
ID_COL, STEP_COL, TARGET_COL, SORT_COL = 'C64','C7','C65','C40'
FMT = '%Y-%m-%d %H:%M:%S.%f'
REF = pd.Timestamp('2018-12-01')          # 저불량 앵커(=minor 이후, 원본 _REF_DATE)

def parse_pm_log(raw):
    # pm_log 엔트리 → (ts, type, verdict). (v1.5) verdict 없으면 major→loud / 그외→quiet 폴백(옛 포맷 호환).
    out=[]
    for e in raw:
        if isinstance(e, dict):
            ts=pd.Timestamp(e["date"]); ty=e.get("type","major")
            vd=e.get("verdict", "loud" if ty=="major" else "quiet")
        else:
            ts=pd.Timestamp(e); ty="major"; vd="loud"          # 옛 날짜-리스트 → major/loud
        out.append((ts,ty,vd))
    return sorted(out, key=lambda x:x[0])

PM_EVENTS = parse_pm_log(json.loads(PM_LOG.read_text(encoding="utf-8")))
print("pm_events:", [(str(t.date()),ty,vd) for t,ty,vd in PM_EVENTS])

# 복원 pkl 로드 → top-10 피처명·파라미터
booster = pickle.load(open(PKL,"rb"))
PKL_FEATS  = booster.feature_name()
PKL_PARAMS = dict(booster.params)
print("pkl feats(10):", PKL_FEATS)
print("pkl params:", {k:PKL_PARAMS[k] for k in ['learning_rate','num_leaves','min_child_samples'] if k in PKL_PARAMS}, "...")


pm_events: [('2018-12-24', 'major', 'loud')]
pkl feats(10): ['is_post_pm', 'post_pm_days', 'days_since_last_pm', 'C33', 'hour_x_c33', 'dslp_x_hour', 'C60_mean_step4', 'hour', 'C59_mean_step4', 'is_special_recipe']
pkl params: {'learning_rate': 0.029017547696366934, 'num_leaves': 175, 'min_child_samples': 5} ...


In [3]:
# ── [Cell 2] 원본 로드 ────────────────────────────────────────────
train_raw = pd.read_csv(DATA/"train_data.csv")
valid_raw = pd.read_csv(DATA/"valid_X.csv")
test_raw  = pd.read_csv(DATA/"test_X.csv")
print("raw:", train_raw.shape, valid_raw.shape, test_raw.shape)


raw: (123614, 65) (20577, 64) (20510, 64)


In [4]:
# ── [Cell 3] 피처 빌더 (원본 preprocessing + feature_engineering 이식) ──
def preprocess(df):
    df = df.copy()
    df[SORT_COL] = pd.to_datetime(df[SORT_COL], format=FMT)
    df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])
    return df.sort_values(SORT_COL).reset_index(drop=True)

def make_fdc_features(df):                       # FDC 23종 × 5통계 × step pivot + C41_max
    numeric = [c for c in FDC_FEATURES if pd.api.types.is_numeric_dtype(df[c])]
    string  = [c for c in FDC_FEATURES if not pd.api.types.is_numeric_dtype(df[c])]
    step_dur = df.groupby([ID_COL,STEP_COL])['C41'].max().rename('C41_max').reset_index()
    agg = df.groupby([ID_COL,STEP_COL])[numeric].agg(AGG_FUNCS)
    agg.columns = ['_'.join(c) for c in agg.columns]
    agg = agg.reset_index().merge(step_dur, on=[ID_COL,STEP_COL])
    wide = agg.pivot(index=ID_COL, columns=STEP_COL)
    wide.columns = [f'{c[0]}_step{int(c[1])}' for c in wide.columns]
    wide = wide.reset_index()
    if string:
        wide = wide.merge(df.groupby(ID_COL)[string].first().reset_index(), on=ID_COL)
    return wide

def _time_feats(dates, pm_events):   # 그룹 A 시간/레짐 (v1.5 verdict) — pm_events=[(ts,type,verdict)] sorted
    dslp, is_post, is_high, hrd, is_high_t, hrd_t = [], [], [], [], [], []
    for d in dates:
        past = [(t,ty,vd) for t,ty,vd in pm_events if t<=d]
        if past:
            last_ts = past[-1][0]; dd = (d-last_ts).total_seconds()/86400
            dslp.append(dd); is_post.append(1)
            state, last_loud = 0, None                       # verdict 상태기계 + last-loud 앵커
            for t,ty,vd in past:
                if vd == "loud": state, last_loud = 1, t     # loud→상태 갱신·앵커 이동, quiet→유지
            is_high.append(state)
            hrd.append((d-last_loud).total_seconds()/86400 if state==1 and last_loud is not None else 0.0)
            lty = past[-1][1]; iht = 1 if lty=="major" else 0   # (옛 type 규칙 — Cell 4 동치 검증용 그림자)
            is_high_t.append(iht); hrd_t.append(dd*iht)
        else:                                                # 관측 PM 이전(=저불량 앵커)
            dslp.append((d-REF).total_seconds()/86400)
            is_post.append(0); is_high.append(0); is_high_t.append(0); hrd.append(0.0); hrd_t.append(0.0)
    return pd.DataFrame({'days_since_last_pm':dslp,'is_post_pm':is_post,'is_high_regime':is_high,
                         'high_regime_days':hrd,'_is_high_type':is_high_t,'_hrd_type':hrd_t}, index=dates.index)

def make_meta_features(df, pm_events):
    wf = df.groupby(ID_COL)
    meta = wf['C33'].first().reset_index()
    wd = wf[SORT_COL].first().reset_index().rename(columns={SORT_COL:'_date'})
    tf = _time_feats(wd['_date'], pm_events)
    wd = pd.concat([wd, tf], axis=1); wd['hour'] = wd['_date'].dt.hour
    cols = [ID_COL,'days_since_last_pm','is_post_pm','is_high_regime','high_regime_days',
            '_is_high_type','_hrd_type','hour']
    meta = meta.merge(wd[cols], on=ID_COL)
    meta['post_pm_days'] = meta['days_since_last_pm'] * meta['is_post_pm']   # pkl 재현용(last-PM 앵커, 옛 이름)
    meta['dslp_x_hour']  = meta['days_since_last_pm'] * meta['hour']
    meta['hour_x_c33']   = meta['hour'] * meta['C33']
    return meta

def build_features(df_raw, pm_events=PM_EVENTS):
    df  = preprocess(df_raw)
    res = make_fdc_features(df).merge(make_meta_features(df, pm_events), on=ID_COL)
    res['is_special_recipe'] = (res['C6']=='C6_1').astype(int)
    res['C23'] = df.groupby(ID_COL)['C23'].first().values     # M1(C23_te) 재료
    if TARGET_COL in df.columns:
        res = res.merge(df.groupby(ID_COL)[TARGET_COL].first().reset_index(), on=ID_COL)
    return res

print("빌더 정의 완료.")


빌더 정의 완료.


In [5]:
# ── [Cell 4] 피처 테이블 생성 + 정합 assert ───────────────────────
Xtr = build_features(train_raw)
Xva = build_features(valid_raw)
Xte = build_features(test_raw)
assert (len(Xtr),len(Xva),len(Xte)) == (11939,1990,1990), "WF 수 불일치"
core10 = ['is_high_regime','high_regime_days','days_since_last_pm','C33','dslp_x_hour',
          'hour','hour_x_c33','C60_mean_step4','C59_mean_step4','is_special_recipe']
miss = [c for c in core10 if c not in Xtr.columns]
assert not miss, f"코어 피처 누락: {miss}"
# v1.5 동치 assert: verdict 규칙 == 옛 type 규칙 (현 데이터 유일 경계 12-24가 loud+major → 동치)
for nm,D in [('train',Xtr),('valid',Xva),('test',Xte)]:
    assert (D['is_high_regime']==D['_is_high_type']).all(),      f"{nm}: is_high_regime(verdict) ≠ is_high(type)"
    assert np.allclose(D['high_regime_days'], D['_hrd_type']),   f"{nm}: high_regime_days(loud앵커) ≠ hrd(type)"
print("✅ v1.5 동치: is_high_regime(verdict)==is_high(type) & high_regime_days(loud앵커)==hrd(type) — 전 세트")
for D in (Xtr,Xva,Xte): D.drop(columns=['_is_high_type','_hrd_type'], inplace=True)   # 그림자 열 제거
print(f"피처 테이블: train {Xtr.shape} / valid {Xva.shape} / test {Xte.shape}")
# pkl 재현 별칭 값-동일: is_high_regime == is_post_pm, high_regime_days == post_pm_days
eq1 = (Xtr['is_high_regime']==Xtr['is_post_pm']).all()
eq2 = np.allclose(Xtr['high_regime_days'], Xtr['post_pm_days'])
print(f"pkl 별칭 값-동일(경계 1회 loud+major): is_high==is_post {eq1}, hrd==post_pm_days {eq2}")
print(f"  급등 {int(Xtr.is_high_regime.sum())} / 저불량 {int((Xtr.is_high_regime==0).sum())} WF")


✅ v1.5 동치: is_high_regime(verdict)==is_high(type) & high_regime_days(loud앵커)==hrd(type) — 전 세트
피처 테이블: train (11939, 569) / valid (1990, 568) / test (1990, 568)
pkl 별칭 값-동일(경계 1회 loud+major): is_high==is_post True, hrd==post_pm_days True
  급등 8247 / 저불량 3692 WF


In [6]:
# ── [Cell 5] M0a — 복원 pkl로 valid/test 직접 예측 (정합 검증) ─────
def rmse_vs_answer(res, ans_path):
    ans = pd.read_csv(ans_path); ans[ID_COL]=ans[ID_COL].astype(str)
    r = res.copy(); r[ID_COL]=r[ID_COL].astype(str)
    pred = booster.predict(r[PKL_FEATS])            # pkl 학습 피처명(is_post_pm 등)으로 예측
    m = r[[ID_COL]].assign(pred=pred).merge(ans.rename(columns={TARGET_COL:'true'}), on=ID_COL)
    return float(np.sqrt(np.mean((m['true']-m['pred'])**2))), len(m)

va_rmse, nva = rmse_vs_answer(Xva, ANS/"valid_Y_answer.csv")
te_rmse, nte = rmse_vs_answer(Xte, ANS/"test_Y_answer.csv")
print(f"[M0a] valid RMSE = {va_rmse:.3f} (n={nva})  |  test RMSE = {te_rmse:.3f} (n={nte})")
G1a = abs(va_rmse-38.3) <= 1.5
print("🟢 G1(a) 통과 — 피처 테이블 정합 (valid≈38.3)" if G1a else "🔴 G1(a) 실패 — diff 디버깅(dslp·pm_date·C59/C60 NaN·피처명)")
print("  → 다음: M0b(동일 10피처·복원 파라미터로 wafer 5-fold 재학습, valid≤42)")


[M0a] valid RMSE = 37.971 (n=1990)  |  test RMSE = 39.041 (n=1990)
🟢 G1(a) 통과 — 피처 테이블 정합 (valid≈38.3)
  → 다음: M0b(동일 10피처·복원 파라미터로 wafer 5-fold 재학습, valid≤42)


In [7]:
# ── [Cell 6] P1 스모크 (v1.5 신세계관 3종) — pm_log 1줄 추가로 자동 대응 + loud앵커 버그 차단 ──
def _build_with(extra):   # pm_log 사본에 가상 이벤트 1줄 추가 → valid 재빌드
    ev = parse_pm_log(json.loads(PM_LOG.read_text(encoding="utf-8")) + [extra])
    return build_features(valid_raw, ev)
base = Xva[[ID_COL,'days_since_last_pm','is_high_regime','high_regime_days']].rename(
        columns={'days_since_last_pm':'dslp0','is_high_regime':'ih0','high_regime_days':'hrd0'})
CUT = 27.3   # 12-24 기준 경과일 27일↑ = 1-20 이후 WF 식별

# (a) 고불량 중 quiet minor: is_high 1유지, hrd 연속(리셋 금지 ← loud앵커), dslp만 리셋
A = _build_with({"date":"2019-01-20","type":"minor","verdict":"quiet"}).merge(base, on=ID_COL)
Aa = A[A.dslp0 > CUT]
a_ok = (Aa.is_high_regime==1).all() and np.allclose(Aa.high_regime_days, Aa.hrd0) and (Aa.days_since_last_pm < Aa.dslp0-0.5).all()
print(f"(a) quiet minor 1-20 → 이후 {len(Aa)}WF: is_high 1유지 & hrd 연속(리셋X) & dslp만 리셋 = {'✅' if a_ok else '❌'}")

# (b) 저불량 중 quiet major: type이 major여도 verdict quiet → 점프 없음(is_high 0유지)
B = _build_with({"date":"2018-12-15","type":"major","verdict":"quiet"}).merge(
      Xva[[ID_COL,'is_high_regime']].rename(columns={'is_high_regime':'ih0'}), on=ID_COL)
Bb = B[B.ih0==0]
b_ok = (Bb.is_high_regime==0).all()
print(f"(b) quiet major 12-15 → 저불량 {len(Bb)}WF: is_high 0유지(quiet=점프 없음) = {'✅' if b_ok else '❌'}")

# (c) 신규 loud major: is_high 갱신, hrd 새 loud 앵커에서 0부터 재시작(hrd==dslp)
C = _build_with({"date":"2019-01-20","type":"major","verdict":"loud"}).merge(base, on=ID_COL)
Cc = C[C.dslp0 > CUT]
c_ok = (Cc.is_high_regime==1).all() and np.allclose(Cc.high_regime_days, Cc.days_since_last_pm)
print(f"(c) loud major 1-20 → 이후 {len(Cc)}WF: is_high 갱신 & hrd 새 loud앵커 재시작(hrd==dslp) = {'✅' if c_ok else '❌'}")

print("  ✅ P1 — pm_log 1줄(date+type+verdict) 추가만으로 전 피처 자동 대응. "
      "(a)가 loud앵커 수정의 '가짜 ~1400 스파이크' 차단을 실증(구 dslp앵커면 hrd=0 오독). 예측-밴드 정밀검증은 M8(§9.4)." if (a_ok and b_ok and c_ok)
      else "  ❌ 스모크 실패 — 빌더 점검")


(a) quiet minor 1-20 → 이후 673WF: is_high 1유지 & hrd 연속(리셋X) & dslp만 리셋 = ✅
(b) quiet major 12-15 → 저불량 606WF: is_high 0유지(quiet=점프 없음) = ✅
(c) loud major 1-20 → 이후 673WF: is_high 갱신 & hrd 새 loud앵커 재시작(hrd==dslp) = ✅
  ✅ P1 — pm_log 1줄(date+type+verdict) 추가만으로 전 피처 자동 대응. (a)가 loud앵커 수정의 '가짜 ~1400 스파이크' 차단을 실증(구 dslp앵커면 hrd=0 오독). 예측-밴드 정밀검증은 M8(§9.4).


## Phase 4 — M0b(재학습) + M-T(시간-only 대조군)

M0a가 "우리 피처 = 원본 피처"를 증명했으니, 이제 **우리 손으로 재학습**해 v8의 CV/valid를 확정한다(**G1 완결**).

- **M0b**: 코어 10피처 + 복원 파라미터로 **wafer 5-fold OOF CV** + full-train(원본 best_iteration 705라운드) → valid/test.
- **M-T**: 시간/레짐 7피처만(센서 3종 `C60/C59_mean_step4`·`is_special_recipe` 제외) 대조군. 이후 모든 마일스톤에서 **"센서 기여 = M-T − 모델"**로 센서 증분을 추적한다(PLAN §8.4 — "시간만 남으면 v5 합칠 의미 없다" 우려 대응).
- **CV 스킴**: 피처 테이블은 **wafer 1행**(고유 C64=행수 11,939) → wafer-level `KFold(5, shuffle, seed42)`. 각 wafer가 단일 그룹이라 이 단계에선 `GroupKFold(C64)`와 목적이 같고, **row-level(M3+)부터** 그룹이 비자명해지므로 그때 `GroupKFold(C64)`로 전환한다.


In [8]:
# ── [Cell 7] M0b — 코어 10피처 재학습 (wafer 5-fold OOF CV + full-train) ──
from sklearn.model_selection import KFold
y = Xtr[TARGET_COL].to_numpy(float)

# 복원 파라미터(pkl) 채택 — Cell 1 PKL_PARAMS에서 학습 하이퍼파라미터
M8_PARAMS = dict(objective='regression', metric='rmse',
    learning_rate   =PKL_PARAMS.get('learning_rate', 0.029017547696366934),
    num_leaves      =PKL_PARAMS.get('num_leaves', 175),
    min_child_samples=PKL_PARAMS.get('min_child_samples', 5),
    feature_fraction=PKL_PARAMS.get('feature_fraction', 0.6324704159196377),
    bagging_fraction=PKL_PARAMS.get('bagging_fraction', 0.864012693783303),
    bagging_freq    =PKL_PARAMS.get('bagging_freq', 7),
    lambda_l1       =PKL_PARAMS.get('lambda_l1', 5.04154328625296),
    lambda_l2       =PKL_PARAMS.get('lambda_l2', 0.024814259264649002),
    min_split_gain  =PKL_PARAMS.get('min_split_gain', 0.2573073648505903),
    verbose=-1, seed=42)
BEST_ROUNDS = 705    # 원본 pkl best_iteration(복원 meta) → full-train 학습 예산

def wafer_cv_oof(feats, params=M8_PARAMS):
    X = Xtr[feats]; oof = np.full(len(y), np.nan)
    for a, b in KFold(5, shuffle=True, random_state=42).split(X):
        d  = lgb.Dataset(X.iloc[a], y[a]); dv = lgb.Dataset(X.iloc[b], y[b], reference=d)
        m  = lgb.train(params, d, num_boost_round=5000, valid_sets=[dv],
                       callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = m.predict(X.iloc[b], num_iteration=m.best_iteration)
    return float(np.sqrt(np.mean((y-oof)**2)))

def full_train(feats, params=M8_PARAMS, rounds=BEST_ROUNDS):
    return lgb.train(params, lgb.Dataset(Xtr[feats], y), num_boost_round=rounds)

def rmse_pred(model, res, feats, ans_path):
    ans = pd.read_csv(ans_path); ans[ID_COL] = ans[ID_COL].astype(str)
    r = res.copy(); r[ID_COL] = r[ID_COL].astype(str)
    p = model.predict(r[feats])
    m = r[[ID_COL]].assign(p=p).merge(ans.rename(columns={TARGET_COL:'t'}), on=ID_COL)
    return float(np.sqrt(np.mean((m['t']-m['p'])**2)))

cv_m0b  = wafer_cv_oof(core10)
mdl_m0b = full_train(core10)
va_m0b  = rmse_pred(mdl_m0b, Xva, core10, ANS/"valid_Y_answer.csv")
te_m0b  = rmse_pred(mdl_m0b, Xte, core10, ANS/"test_Y_answer.csv")
print(f"[M0b] 10피처 재학습   CV(wafer,OOF)={cv_m0b:.3f}  valid={va_m0b:.3f}  test={te_m0b:.3f}")
G1 = (va_m0b <= 42) and (cv_m0b <= 44)
print("🟢 G1 완결 — 재학습 valid≤42 & CV~41 확정" if G1 else "🔴 G1 미완 — 파라미터/피처 점검")
print("  → 다음: M1(+C23_te) — fold 내부 OOF 타깃 인코딩")


[M0b] 10피처 재학습   CV(wafer,OOF)=40.347  valid=38.399  test=38.912
🟢 G1 완결 — 재학습 valid≤42 & CV~41 확정
  → 다음: M1(+C23_te) — fold 내부 OOF 타깃 인코딩


In [9]:
# ── [Cell 8] M-T — 시간-only 대조군 (센서 3종 제외) + 센서 기여 ────
TIME7 = ['is_high_regime','high_regime_days','days_since_last_pm','C33','dslp_x_hour','hour','hour_x_c33']
assert set(core10) - set(TIME7) == {'C60_mean_step4','C59_mean_step4','is_special_recipe'}, "M-T=코어10−센서3 이어야"
cv_mt  = wafer_cv_oof(TIME7)
mdl_mt = full_train(TIME7)
va_mt  = rmse_pred(mdl_mt, Xva, TIME7, ANS/"valid_Y_answer.csv")
te_mt  = rmse_pred(mdl_mt, Xte, TIME7, ANS/"test_Y_answer.csv")
print(f"[M-T] 시간 7피처       CV(wafer,OOF)={cv_mt:.3f}  valid={va_mt:.3f}  test={te_mt:.3f}")
print(f"[센서 기여] CV {cv_mt:.2f}(M-T) − {cv_m0b:.2f}(M0b) = {cv_mt-cv_m0b:+.2f}pt"
      f"  |  valid {va_mt-va_m0b:+.2f}pt  |  test {te_mt-te_m0b:+.2f}pt")
print("  → 센서(C59/C60 집계 + 레시피)가 시간축 위에 유의미하게 +기여 → v5 센서 결합 의미 있음(PLAN §8.4)")


[M-T] 시간 7피처       CV(wafer,OOF)=44.260  valid=44.622  test=44.149
[센서 기여] CV 44.26(M-T) − 40.35(M0b) = +3.91pt  |  valid +6.22pt  |  test +5.24pt
  → 센서(C59/C60 집계 + 레시피)가 시간축 위에 유의미하게 +기여 → v5 센서 결합 의미 있음(PLAN §8.4)


## M1 — +C23_te (그룹 C, 실험으로 채택 판정)

v5의 유일한 실질 발굴이던 **C23 OOF 타깃인코딩**(m=20, fold-내 fit)을 코어 10에 얹어(11피처), **레짐을 통제한 뒤에도 C23에 잔여 신호가 남는지**를 본다. 누수 차단을 위해 **중첩 OOF**로 인코딩한다 — outer-train 행은 내부 5-fold OOF, outer-val 행은 outer-train 전체 통계로 인코딩(전역 fit 금지, PLAN §7.2).

**채택 게이트**: ΔCV = M1 − M0b ≤ **−0.3**이면 채택. E9에서 C23이 시간 클러스터(레짐 프록시)로 판명됐으므로 **기각 예상**(M0가 이미 흡수).


In [10]:
# ── [Cell 9] M1 — +C23_te (11피처, fold-내 중첩 OOF 타깃인코딩 m=20) ──
M_SMOOTH = 20.0
def _fit_te(cat, yv, m=M_SMOOTH):                 # 스무딩 타깃인코딩 fit
    gm = float(np.mean(yv))
    g  = pd.DataFrame({'c':cat.values,'y':yv}).groupby('c')['y'].agg(['sum','count'])
    return (g['sum'] + m*gm) / (g['count'] + m), gm
def _apply_te(cat, enc, gm): return cat.map(enc).fillna(gm).to_numpy(float)

def m1_cv_oof():                                   # outer 5-fold OOF (내부 OOF로 train 인코딩 → 누수 차단)
    X = Xtr[core10]; c23 = Xtr['C23']; oof = np.full(len(y), np.nan)
    for a,b in KFold(5, shuffle=True, random_state=42).split(X):
        te_a = np.full(len(a), np.nan)
        for ia,ib in KFold(5, shuffle=True, random_state=1).split(a):
            enc,gm = _fit_te(c23.iloc[a[ia]], y[a[ia]]); te_a[ib] = _apply_te(c23.iloc[a[ib]], enc, gm)
        enc,gm = _fit_te(c23.iloc[a], y[a]); te_b = _apply_te(c23.iloc[b], enc, gm)   # val=outer-train 전체 통계
        Xa = X.iloc[a].assign(C23_te=te_a); Xb = X.iloc[b].assign(C23_te=te_b)
        d = lgb.Dataset(Xa, y[a]); dv = lgb.Dataset(Xb, y[b], reference=d)
        mm = lgb.train(M8_PARAMS, d, num_boost_round=5000, valid_sets=[dv], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = mm.predict(Xb, num_iteration=mm.best_iteration)
    return float(np.sqrt(np.mean((y-oof)**2)))

te_full = np.full(len(y), np.nan)                  # full-train용 OOF C23_te
for a,b in KFold(5, shuffle=True, random_state=42).split(Xtr):
    enc,gm = _fit_te(Xtr['C23'].iloc[a], y[a]); te_full[b] = _apply_te(Xtr['C23'].iloc[b], enc, gm)
enc_all,gm_all = _fit_te(Xtr['C23'], y)            # 전체 train → valid/test 인코딩

cv_m1  = m1_cv_oof()
mdl_m1 = lgb.train(M8_PARAMS, lgb.Dataset(Xtr[core10].assign(C23_te=te_full), y), num_boost_round=BEST_ROUNDS)
va_m1  = rmse_pred(mdl_m1, Xva.assign(C23_te=_apply_te(Xva['C23'],enc_all,gm_all)), core10+['C23_te'], ANS/"valid_Y_answer.csv")
te_m1  = rmse_pred(mdl_m1, Xte.assign(C23_te=_apply_te(Xte['C23'],enc_all,gm_all)), core10+['C23_te'], ANS/"test_Y_answer.csv")
dCV = cv_m1 - cv_m0b
print(f"[M1] +C23_te 11피처  CV(wafer,OOF)={cv_m1:.3f}  valid={va_m1:.3f}  test={te_m1:.3f}")
print(f"[채택 게이트] ΔCV = {cv_m1:.3f}(M1) − {cv_m0b:.3f}(M0b) = {dCV:+.3f}  → "
      + ("✅ 채택 (레짐 통제 후에도 C23 잔여 신호)" if dCV<=-0.3 else "❌ 기각 — C23_te는 레짐 프록시(M0가 이미 흡수), 잔여 신호 없음"))
print("  → 다음: M2(센서 집계 풀 gain 재선별, C25 우선)")


[M1] +C23_te 11피처  CV(wafer,OOF)=41.028  valid=38.783  test=39.197
[채택 게이트] ΔCV = 41.028(M1) − 40.347(M0b) = +0.681  → ❌ 기각 — C23_te는 레짐 프록시(M0가 이미 흡수), 잔여 신호 없음
  → 다음: M2(센서 집계 풀 gain 재선별, C25 우선)


## M2 — 센서 집계 풀 투입 → gain 재선별 (그룹 C, 채택 게이트)

코어 10 + **그룹 C 집계 풀 전체**(23센서×5통계×step + C41_max)를 넣고, 1차 학습의 **gain으로 재선별**(2-pass, FS §6)한다. 레짐이 잡힌 상태에서 gain 랭킹이 재편돼 **더 나은 조합**이 나오는지, 특히 EDA에서 급등 내 +0.459였던 **C25**가 부상하는지 본다.

**채택 게이트**: 최고 TOP_N 조합이 M0b 대비 ΔCV ≤ **−0.3** → 채택. (보조로 "코어10 + 신규센서 K개"도 확인.)


In [11]:
# ── [Cell 10] M2 — 센서 집계 풀 투입 → gain 재선별 (2-pass) ──
NON_FEAT = {ID_COL, TARGET_COL, 'C23', 'C6', 'is_post_pm', 'post_pm_days'}   # id·타깃·문자열·pkl별칭 제외
POOL = [c for c in Xtr.columns if c not in NON_FEAT and pd.api.types.is_numeric_dtype(Xtr[c])]
print(f"[M2] 센서 풀 포함 전체 POOL = {len(POOL)}개 (코어10 + 센서 집계 + 시간)")

def cv_oof_gain(feats):                       # wafer 5-fold OOF + fold-평균 gain
    X = Xtr[feats]; oof = np.full(len(y), np.nan); g = np.zeros(len(feats))
    for a, b in KFold(5, shuffle=True, random_state=42).split(X):
        d = lgb.Dataset(X.iloc[a], y[a]); dv = lgb.Dataset(X.iloc[b], y[b], reference=d)
        mm = lgb.train(M8_PARAMS, d, num_boost_round=5000, valid_sets=[dv], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = mm.predict(X.iloc[b], num_iteration=mm.best_iteration); g += mm.feature_importance('gain')
    return float(np.sqrt(np.mean((y-oof)**2))), dict(zip(feats, g/5))

# pass1 — 전체 풀 CV + gain 랭킹
pool_cv, gain = cv_oof_gain(POOL); ranked = sorted(gain, key=gain.get, reverse=True); tot = sum(gain.values()) or 1
"""print(f"[M2 pass1] 전체 풀 CV={pool_cv:.3f} (풀 통째론 노이즈)  gain TOP10(상대%):")
for i, f in enumerate(ranked[:10], 1):
    tag = ' ★C25' if f.startswith('C25_') else (' ·신센서' if f not in core10 else '')
    print(f"   {i:2d}. {f:20s} {100*gain[f]/tot:5.1f}%{tag}")"""
print(f"[M2 pass1] 전체 풀 CV={pool_cv:.3f} (풀 통째론 노이즈)  gain TOP15(상대%):")
for i, f in enumerate(ranked[:15], 1):
    tag = ' ★C25' if f.startswith('C25_') else (' ·신센서' if f not in core10 else '')
    print(f"   {i:2d}. {f:20s} {100*gain[f]/tot:5.1f}%{tag}")
c25 = [f for f in ranked if f.startswith('C25_')]
print(f"   C25 aggregate {len(c25)}개 · 최고 gain 순위 {ranked.index(c25[0])+1 if c25 else '-'}")

# pass2 — gain TOP_N 재선별 스윕
best = (None, 1e9, None)
for N in [10, 12, 15, 20, 25]:
    cvN = wafer_cv_oof(ranked[:N]); print(f"   TOP_{N:2d}  CV={cvN:.3f}  ΔvsM0b={cvN-cv_m0b:+.3f}")
    if cvN < best[1]: best = (N, cvN, ranked[:N])
N, cvB, topB = best

# 보조 probe — 코어10 유지 + gain 상위 신규 센서 K개 추가
new_ranked = [f for f in ranked if f not in core10]
pb = min((wafer_cv_oof(core10 + new_ranked[:K]), K) for K in [3, 5, 8])
print(f"   [probe] 코어10 + 신규센서 최선(+{pb[1]})  CV={pb[0]:.3f}  Δ={pb[0]-cv_m0b:+.3f}")

mdl_m2 = lgb.train(M8_PARAMS, lgb.Dataset(Xtr[topB], y), num_boost_round=BEST_ROUNDS)
va_m2 = rmse_pred(mdl_m2, Xva, topB, ANS/"valid_Y_answer.csv"); te_m2 = rmse_pred(mdl_m2, Xte, topB, ANS/"test_Y_answer.csv")
dCV = cvB - cv_m0b
print(f"[M2 최고] gain TOP_{N}  CV={cvB:.3f}  valid={va_m2:.3f}  test={te_m2:.3f}")
print(f"[채택 게이트] ΔCV = {cvB:.3f} − {cv_m0b:.3f}(M0b) = {dCV:+.3f}  → "
      + ("✅ 채택" if dCV<=-0.3 else "❌ 기각 — 센서 풀 재선별이 코어 10을 못 넘음"))
print("  → C59/C60_mean_step4는 개별 gain 낮아도(~18위) 집단으로 필수(FS §5): gain 선택이 이를 떨궈 악화. C25도 미부상(23위). 코어 10 유지. 다음 M3(row-level).")


[M2] 센서 풀 포함 전체 POOL = 563개 (코어10 + 센서 집계 + 시간)
[M2 pass1] 전체 풀 CV=53.006 (풀 통째론 노이즈)  gain TOP15(상대%):
    1. is_high_regime        50.2%
    2. high_regime_days      27.2%
    3. days_since_last_pm     9.1%
    4. C12_mean_step6         4.4% ·신센서
    5. C33                    1.7%
    6. dslp_x_hour            0.5%
    7. C12_mean_step7         0.5% ·신센서
    8. hour                   0.4%
    9. C4_mean_step4          0.3% ·신센서
   10. C4_max_step1           0.3% ·신센서
   11. hour_x_c33             0.2%
   12. C12_max_step6          0.2% ·신센서
   13. C61_mean_step1         0.1% ·신센서
   14. C60_std_step4          0.1% ·신센서
   15. C62_mean_step1         0.1% ·신센서
   C25 aggregate 25개 · 최고 gain 순위 23
   TOP_10  CV=47.017  ΔvsM0b=+6.670
   TOP_12  CV=46.996  ΔvsM0b=+6.649
   TOP_15  CV=43.937  ΔvsM0b=+3.590
   TOP_20  CV=45.076  ΔvsM0b=+4.729
   TOP_25  CV=45.836  ΔvsM0b=+5.489
   [probe] 코어10 + 신규센서 최선(+3)  CV=41.052  Δ=+0.705
[M2 최고] gain TOP_15  CV=43.937  valid=41.674  test=43.253
[채택 게

## M3 — row-level 결합 (v5 프레임 + 그룹 A/B broadcast)

WF 단위 집계 대신 **행(step) 단위**로 C65를 예측한 뒤 WF 평균으로 되돌린다. v5 row-level 빌더(원본 센서 + WF context broadcast + `row_pos` + `C6`/`C7` 범주형 + C23 OOF 타깃인코딩)에 **그룹 A(레짐/시간) broadcast**와 **행별 `hour_row`**를 얹는다. 레짐 피처가 WF 내 상수라, 행 단위 step×센서 신호가 그 위에서도 남는지 본다.

**공정 비교**: M0b와 **동일한 WF 5-fold 분할**로 OOF를 만들어(행→WF 평균 후 WF-RMSE) ΔCV를 직접 잰다.
**판정 게이트**: ΔCV ≤ **−0.3** → 프레임 교체. 아니면 WF-level(M0b) 확정. **E10**(v5 잔차 vs 레짐 피처 상관≈0)로 **기대 하향** — 저비용 확인 실험.

In [12]:
# ── [Cell 11] M3 — row-level 결합 (v5 프레임 + 그룹 A/B broadcast → WF 평균) ──
# 가설: 행(step) 단위로 C65를 예측한 뒤 WF 평균이, 레짐 피처(WF 상수) 위에서도 이득이 있는가.
# 공정성: M0b와 '동일한 WF 5-fold 분할'로 OOF를 만들어(행→WF 평균 후 WF-RMSE) ΔCV를 직접 비교.
CATS_M3 = ['C6','C7']                                    # LightGBM 범주형 (step×센서 상호작용)
GROUP_A = ['is_high_regime','high_regime_days','days_since_last_pm','C33',
           'dslp_x_hour','hour','hour_x_c33']            # 그룹 A(레짐/시간) — WF 상수 → 각 행에 broadcast
DROP_SENS = set(COLS_TO_DROP) | {'C34','C35','C38','C20','C21','C22',
                                 'C33','C10','C39','C40','C41'}   # 식별/Lot/시간·중복 제외 (C40는 hour_row로만 사용)

def build_rows_m3(df_raw, wf_tab, has_target):
    df = df_raw.copy()
    df[SORT_COL] = pd.to_datetime(df[SORT_COL], format=FMT, errors='coerce')
    df = df.sort_values(SORT_COL).reset_index(drop=True)
    df['hour_row'] = df[SORT_COL].dt.hour.astype(float)          # 행별 시각 (그룹 A hour는 WF-first, 이건 행 단위)
    excl = set([ID_COL,TARGET_COL,'C23','hour_row']) | set(CATS_M3) | DROP_SENS
    sensors = [c for c in df.select_dtypes(include=[np.number]).columns if c not in excl]
    df['row_pos'] = df.groupby(ID_COL).cumcount()                # WF 내 행 순번
    g = df.groupby(ID_COL)
    ctx = g[sensors].agg(['mean','std','min','max']); ctx.columns = [f'{c}_wf_{s}' for c,s in ctx.columns]
    ctx['wf_nrows'] = g.size(); ctx_cols = list(ctx.columns)     # WF 전역 context broadcast (그룹 B 센서 포함)
    df = df.merge(ctx.reset_index(), on=ID_COL, how='left')
    df = df.drop(columns=[c for c in GROUP_A if c in df.columns]) # raw C33 등 GROUP_A와 겹치는 원본열 제거(충돌 방지)
    df = df.merge(wf_tab[[ID_COL]+GROUP_A], on=ID_COL, how='left')# 그룹 A broadcast
    for c in CATS_M3: df[c] = df[c].astype('category')
    feat_cols = sensors + ctx_cols + ['row_pos','hour_row'] + GROUP_A + CATS_M3
    keep = [ID_COL] + feat_cols + ['C23'] + ([TARGET_COL] if has_target else [])
    return df[keep].copy(), feat_cols

tr_rows, R_FEATS = build_rows_m3(train_raw, Xtr, True)
va_rows, _       = build_rows_m3(valid_raw, Xva, False)
te_rows, _       = build_rows_m3(test_raw,  Xte, False)
print(f"[M3] row-level 피처 {len(R_FEATS)}개 | train rows {len(tr_rows):,}")

# 행→WF fold 매핑 (M0b와 '동일한' WF 5-fold 분할 → ΔCV 직접 비교 가능)
wf_ids = Xtr[ID_COL].to_numpy(); wf2fold = {}
for k,(a,b) in enumerate(KFold(5, shuffle=True, random_state=42).split(wf_ids)):
    for wid in wf_ids[b]: wf2fold[wid] = k
tr_rows['_fold'] = tr_rows[ID_COL].map(wf2fold)

def te_fit(frame, m=20.0):                                       # v5식 스무딩 타깃인코딩 (fold-내 train만 fit → 누수 차단)
    prior = frame[TARGET_COL].mean()
    agg = frame.groupby('C23')[TARGET_COL].agg(['mean','count'])
    return ((agg['mean']*agg['count'] + prior*m)/(agg['count']+m)), prior

ROW_PARAMS = dict(objective='regression', metric='rmse', learning_rate=0.03, num_leaves=127,
    min_child_samples=50, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
    lambda_l1=0.5, lambda_l2=1.0, verbose=-1, seed=42)           # v5 row-level 레시피(행 단위에 맞춤)
USE = R_FEATS + ['C23_te']
oof_row = np.full(len(tr_rows), np.nan); va_acc = np.zeros(len(va_rows)); te_acc = np.zeros(len(te_rows))
for k in range(5):
    tri = (tr_rows['_fold']!=k).to_numpy(); vai = (tr_rows['_fold']==k).to_numpy()
    tf = tr_rows[tri].copy(); vf = tr_rows[vai].copy()
    enc, prior = te_fit(tf)
    tf['C23_te'] = tf['C23'].map(enc).fillna(prior); vf['C23_te'] = vf['C23'].map(enc).fillna(prior)
    vak = va_rows.copy(); tek = te_rows.copy()
    vak['C23_te'] = vak['C23'].map(enc).fillna(prior); tek['C23_te'] = tek['C23'].map(enc).fillna(prior)
    d  = lgb.Dataset(tf[USE], tf[TARGET_COL], categorical_feature=CATS_M3)
    dv = lgb.Dataset(vf[USE], vf[TARGET_COL], reference=d)
    m  = lgb.train(ROW_PARAMS, d, num_boost_round=4000, valid_sets=[dv],
                   callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_row[vai] = m.predict(vf[USE], num_iteration=m.best_iteration)
    va_acc += m.predict(vak[USE], num_iteration=m.best_iteration) / 5
    te_acc += m.predict(tek[USE], num_iteration=m.best_iteration) / 5
    print(f"   fold{k+1} best_iter={m.best_iteration}")

def wf_mean(rows, pred):
    return pd.DataFrame({ID_COL:rows[ID_COL].values,'p':pred}).groupby(ID_COL)['p'].mean()
oof_wf = wf_mean(tr_rows, oof_row); y_wf = Xtr.set_index(ID_COL)[TARGET_COL]
cv_m3  = float(np.sqrt(np.mean((y_wf.loc[oof_wf.index].to_numpy() - oof_wf.to_numpy())**2)))
def wf_rmse(rows, pred, ans_path):
    ans = pd.read_csv(ans_path); ans[ID_COL] = ans[ID_COL].astype(str); p = wf_mean(rows, pred)
    pr = pd.DataFrame({ID_COL:p.index.astype(str), 'p':p.values})
    mm = pr.merge(ans.rename(columns={TARGET_COL:'t'}), on=ID_COL)
    return float(np.sqrt(np.mean((mm['t']-mm['p'])**2)))
va_m3 = wf_rmse(va_rows, va_acc, ANS/"valid_Y_answer.csv")
te_m3 = wf_rmse(te_rows, te_acc, ANS/"test_Y_answer.csv")
dCV = cv_m3 - cv_m0b
print(f"[M3] row-level 결합  CV(wafer,OOF)={cv_m3:.3f}  valid={va_m3:.3f}  test={te_m3:.3f}")
print(f"[판정 게이트] ΔCV = {cv_m3:.3f}(M3) − {cv_m0b:.3f}(M0b) = {dCV:+.3f}  → "
      + ("✅ 채택 — row-level 프레임 교체" if dCV<=-0.3 else "❌ 기각 — WF-level(M0b) 확정, row-level 이득 없음(E10 예측대로)"))
print("  → 다음: M4(TOP_N·그룹 ablation·시간 dedup → G2)")

[M3] row-level 피처 157개 | train rows 123,614
   fold1 best_iter=970
   fold2 best_iter=347
   fold3 best_iter=758
   fold4 best_iter=807
   fold5 best_iter=570
[M3] row-level 결합  CV(wafer,OOF)=50.868  valid=49.273  test=48.875
[판정 게이트] ΔCV = 50.868(M3) − 40.347(M0b) = +10.521  → ❌ 기각 — WF-level(M0b) 확정, row-level 이득 없음(E10 예측대로)
  → 다음: M4(TOP_N·그룹 ablation·시간 dedup → G2)


## M4 — 피처셋 최종 확정 (블록 ablation · 시간 dedup · TOP_N 스윕 → G2)

코어 10이 M1(+C23_te)·M2(센서 풀)·M3(row-level)을 **모두** 이겼다. M4는 이 코어 10을 **증거로 못박는다** — (1) **블록 ablation**: 레짐(A)·센서(B)를 통째 빼서 각 블록의 필수성 확인(유지 규칙 ΔCV ≥ +0.3), (2) **시간 dedup**: 시간 7종의 실질 자유도(최소셋 vs 전체), (3) **TOP_N 스윕**: gain-greedy 선택(M2 랭킹)이 수기 코어 10을 못 넘는지.

**G2**: 두 블록 모두 필수 & TOP_N 최소가 10 근방 → 코어 10 확정.

In [13]:
# ── [Cell 12] M4 — 피처셋 최종 확정 (블록 ablation · 시간 dedup · TOP_N 스윕 → G2) ──
GA = ['is_high_regime','high_regime_days','days_since_last_pm','C33',
      'dslp_x_hour','hour','hour_x_c33']                      # 그룹 A 레짐/시간 7
GB = ['C60_mean_step4','C59_mean_step4','is_special_recipe']  # 그룹 B 센서/레시피 3
assert sorted(GA+GB) == sorted(core10), "GA+GB가 코어10과 일치해야"

# (1) 블록 ablation — 블록을 통째 제거했을 때 CV 악화폭(=필수성). 유지 규칙 ΔCV ≥ +0.3
print(f"── (1) 블록 ablation (baseline M0b CV = {cv_m0b:.3f}) ──")
abl = {'core10 (GA+GB)': core10, '−GA 레짐제거 (GB만 3)': GB, '−GB 센서제거 (GA만 7=M-T)': GA}
abl_cv = {}
for name, feats in abl.items():
    c = wafer_cv_oof(feats); abl_cv[name] = c; d = c - cv_m0b
    tag = '' if name.startswith('core10') else ('  ← 필수(유지)' if d >= 0.3 else '  ← 기여 미미(재검토)')
    print(f"   {name:26s} ({len(feats):2d}f)  CV={c:.3f}  ΔvsM0b={d:+.3f}{tag}")

# (2) 시간 블록 dedup — 시간 7종의 실질 자유도. 최소셋 {is_high_regime, high_regime_days, hour}
print("── (2) 시간 dedup ──")
TIME_MIN = ['is_high_regime','high_regime_days','hour']
for name, feats in {'GB+시간7 (=core10)': GB+GA, 'GB+시간최소3': GB+TIME_MIN,
                    '시간7만 (M-T)': GA, '시간최소3만': TIME_MIN}.items():
    c = wafer_cv_oof(feats)
    print(f"   {name:20s} ({len(feats):2d}f)  CV={c:.3f}  ΔvsM0b={c-cv_m0b:+.3f}")

# (3) TOP_N 스윕 — gain-greedy(M2 ranked)가 수기 코어10을 넘는지
print("── (3) TOP_N 스윕 (M2 gain 랭킹) ──")
best = (None, 1e9)
for N in [8, 10, 12, 15, 20]:
    c = wafer_cv_oof(ranked[:N])
    if c < best[1]: best = (N, c)
    print(f"   TOP_{N:2d}  CV={c:.3f}  ΔvsM0b={c-cv_m0b:+.3f}")
print(f"   [참조] 수기 코어10 CV={cv_m0b:.3f}   |  gain-greedy 최소 = TOP_{best[0]} CV={best[1]:.3f}")

# ── G2 판정 ──
d_GA = abl_cv['−GA 레짐제거 (GB만 3)'] - cv_m0b
d_GB = abl_cv['−GB 센서제거 (GA만 7=M-T)'] - cv_m0b
G2 = (d_GA >= 0.3) and (d_GB >= 0.3) and (best[1] >= cv_m0b - 0.3)
print("── G2 판정 ──")
print(f"   · 레짐(GA) 제거 ΔCV {d_GA:+.2f} → {'필수' if d_GA>=0.3 else '재검토'}")
print(f"   · 센서(GB) 제거 ΔCV {d_GB:+.2f} → {'집단 필수(§8.4)' if d_GB>=0.3 else '재검토'}")
print(f"   · gain-greedy가 코어10 못 넘음: {best[1] >= cv_m0b - 0.3}")
print("🟢 G2 통과 — 코어 10 WF-level 확정, M5(모델 벤치)로" if G2 else "🟡 G2 재검토 — 위 수치로 조합 재조정")
print("  → 다음: M5a 모델 후보 벤치(LGBM/XGB/CatBoost/HistGB…) → M5b 상위 2개 Optuna")

── (1) 블록 ablation (baseline M0b CV = 40.347) ──
   core10 (GA+GB)             (10f)  CV=40.347  ΔvsM0b=+0.000
   −GA 레짐제거 (GB만 3)           ( 3f)  CV=259.229  ΔvsM0b=+218.882  ← 필수(유지)
   −GB 센서제거 (GA만 7=M-T)       ( 7f)  CV=44.260  ΔvsM0b=+3.913  ← 필수(유지)
── (2) 시간 dedup ──
   GB+시간7 (=core10)     (10f)  CV=40.300  ΔvsM0b=-0.047
   GB+시간최소3             ( 6f)  CV=48.355  ΔvsM0b=+8.008
   시간7만 (M-T)           ( 7f)  CV=44.260  ΔvsM0b=+3.913
   시간최소3만               ( 3f)  CV=50.414  ΔvsM0b=+10.067
── (3) TOP_N 스윕 (M2 gain 랭킹) ──
   TOP_ 8  CV=46.967  ΔvsM0b=+6.620
   TOP_10  CV=47.017  ΔvsM0b=+6.670
   TOP_12  CV=46.996  ΔvsM0b=+6.649
   TOP_15  CV=43.937  ΔvsM0b=+3.590
   TOP_20  CV=45.076  ΔvsM0b=+4.729
   [참조] 수기 코어10 CV=40.347   |  gain-greedy 최소 = TOP_15 CV=43.937
── G2 판정 ──
   · 레짐(GA) 제거 ΔCV +218.88 → 필수
   · 센서(GB) 제거 ΔCV +3.91 → 집단 필수(§8.4)
   · gain-greedy가 코어10 못 넘음: True
🟢 G2 통과 — 코어 10 WF-level 확정, M5(모델 벤치)로
  → 다음: M5a 모델 후보 벤치(LGBM/XGB/CatBoost/HistGB…) → M5b 상위 2개 Optu

# M5a — 모델 후보 벤치마크

## xgboost·catboost 설치 확인

In [15]:
# ── [Cell 12.5] M5 환경: xgboost·catboost 설치 확인 (미설치 시 벤치에서 자동 제외) ──
import importlib
for lib in ['xgboost','catboost']:
    try:
        m = importlib.import_module(lib); print(f"✅ {lib} {m.__version__}")
    except ImportError:
        print(f"⚠️  {lib} 미설치 →  아래를 터미널(venv 활성) 또는 셀에서 1회 실행")
        print(f"     %pip install {lib}")

✅ xgboost 3.3.0
✅ catboost 1.2.10


## M5a — 벤치 하니스 + 후보 7종 등록

In [16]:
# ── [Cell 13] M5a — 모델 후보 벤치 하니스 + 후보 등록 (§8.3) ──
import time
from sklearn.model_selection import KFold

# 관찰 트랙 (§8.5 동결 상수 — 재유도 금지)
F_C10 = core10
F_T15 = ['is_high_regime','high_regime_days','days_since_last_pm','C12_mean_step6','C33',
         'dslp_x_hour','C12_mean_step7','hour','C4_mean_step4','C4_max_step1','hour_x_c33',
         'C12_max_step6','C61_mean_step1','C60_std_step4','C62_mean_step1']
F_P3  = core10 + ['C12_mean_step6','C12_mean_step7','C4_mean_step4']
TRACKS = {'F-C10': F_C10, 'F-T15': F_T15, 'F-P3': F_P3}
for tn, ff in TRACKS.items():
    miss = [c for c in ff if c not in Xtr.columns]
    assert not miss, f"{tn} 트랙 피처 누락: {miss}"

FOLDS = list(KFold(5, shuffle=True, random_state=42).split(Xtr))   # M0~M4와 동일 wafer 5-fold

def run_candidate(fit_pred, feats):
    """fit_pred(Xa_df, ya, Xb_df, yb) -> pred_b. 동일 FOLDS로 OOF 생성 → WF-RMSE·fold std·시간."""
    Xf = Xtr[feats]; oof = np.full(len(y), np.nan); fr = []
    t0 = time.perf_counter()
    for a, b in FOLDS:
        pb = np.asarray(fit_pred(Xf.iloc[a], y[a], Xf.iloc[b], y[b]), float)
        oof[b] = pb; fr.append(float(np.sqrt(np.mean((y[b]-pb)**2))))
    dt = time.perf_counter() - t0
    cv = float(np.sqrt(np.mean((y-oof)**2)))
    return dict(cv=cv, std=float(np.std(fr)), time=dt, oof=oof)

CANDS = {}   # 후보 레지스트리 (미설치 라이브러리는 건너뜀)

def _fp_lgbm(Xa, ya, Xb, yb):                     # 1) LightGBM 기준 (M8_PARAMS+early stop → cv_m0b 재현)
    d = lgb.Dataset(Xa, ya); dv = lgb.Dataset(Xb, yb, reference=d)
    m = lgb.train(M8_PARAMS, d, num_boost_round=5000, valid_sets=[dv],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    return m.predict(Xb, num_iteration=m.best_iteration)
CANDS['LGBM'] = _fp_lgbm

try:                                              # 2) XGBoost (hist, depth-wise, NaN 네이티브)
    import xgboost as xgb
    def _fp_xgb(Xa, ya, Xb, yb):
        m = xgb.XGBRegressor(n_estimators=5000, learning_rate=0.05, max_depth=6,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                             tree_method='hist', eval_metric='rmse', early_stopping_rounds=100,
                             n_jobs=-1, random_state=42)
        m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
        return m.predict(Xb)
    CANDS['XGBoost'] = _fp_xgb
except Exception as e:
    print("  · XGBoost 제외:", e)

try:                                              # 3) CatBoost (ordered boosting)
    from catboost import CatBoostRegressor
    def _fp_cat(Xa, ya, Xb, yb):
        m = CatBoostRegressor(iterations=5000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
                              loss_function='RMSE', random_seed=42, od_type='Iter', od_wait=100,
                              allow_writing_files=False, verbose=False)
        m.fit(Xa, ya, eval_set=(Xb, yb), use_best_model=True)
        return m.predict(Xb)
    CANDS['CatBoost'] = _fp_cat
except Exception as e:
    print("  · CatBoost 제외:", e)

from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
def _fp_hgb(Xa, ya, Xb, yb):                      # 4) HistGB (NaN 네이티브, 내부 early stop=누수 없음)
    m = HistGradientBoostingRegressor(random_state=42, early_stopping=True)
    m.fit(Xa, ya); return m.predict(Xb)
CANDS['HistGB'] = _fp_hgb

def _fp_et(Xa, ya, Xb, yb):                       # 5) ExtraTrees (배깅; NaN→fold내 중앙값 대치)
    imp = SimpleImputer(strategy='median').fit(Xa)
    m = ExtraTreesRegressor(n_estimators=500, n_jobs=-1, random_state=42)
    m.fit(imp.transform(Xa), ya); return m.predict(imp.transform(Xb))
CANDS['ExtraTrees'] = _fp_et

from sklearn.linear_model import RidgeCV          # 6) 정칙화 선형 + 스플라인/주기 인코딩 (fold내 fit)
from sklearn.preprocessing import StandardScaler, SplineTransformer, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
def _cyc(period):
    return FunctionTransformer(lambda X: np.column_stack(
        [np.sin(2*np.pi*np.asarray(X, float).ravel()/period),
         np.cos(2*np.pi*np.asarray(X, float).ravel()/period)]))
def _fp_lin(Xa, ya, Xb, yb):
    feats = list(Xa.columns)
    spl  = [c for c in ['days_since_last_pm','high_regime_days'] if c in feats]
    hrc  = [c for c in ['hour'] if c in feats]
    rest = [c for c in feats if c not in spl+hrc]
    parts = []
    if spl:  parts.append(('spl', Pipeline([('imp',SimpleImputer(strategy='median')),
                                            ('s',SplineTransformer(n_knots=4, degree=3))]), spl))
    if hrc:  parts.append(('cyc', Pipeline([('imp',SimpleImputer(strategy='median')),
                                            ('c',_cyc(24.0))]), hrc))
    if rest: parts.append(('num', Pipeline([('imp',SimpleImputer(strategy='median')),
                                            ('sc',StandardScaler())]), rest))
    pipe = Pipeline([('ct',ColumnTransformer(parts)),
                     ('rg',RidgeCV(alphas=np.logspace(-3,3,13)))])
    pipe.fit(Xa, ya); return pipe.predict(Xb)
CANDS['Ridge+spline'] = _fp_lin

def _fp_seg(Xa, ya, Xb, yb):                      # 7) 레짐 세그먼트 LGBM (is_high_regime 라우팅, 2모델)
    pred = np.full(len(Xb), np.nan)
    ha = Xa['is_high_regime'].to_numpy() > 0.5; hb = Xb['is_high_regime'].to_numpy() > 0.5
    for ma, mb in [(~ha, ~hb), (ha, hb)]:
        if mb.sum() == 0: continue
        d = lgb.Dataset(Xa, ya) if ma.sum() < 50 else lgb.Dataset(Xa.iloc[ma], ya[ma])  # 소표본 폴백
        m = lgb.train(M8_PARAMS, d, num_boost_round=BEST_ROUNDS)
        pred[mb] = m.predict(Xb.iloc[mb])
    return pred
CANDS['SegLGBM'] = _fp_seg

print(f"[M5a] 등록 후보 {len(CANDS)}종: {list(CANDS)}")

[M5a] 등록 후보 7종: ['LGBM', 'XGBoost', 'CatBoost', 'HistGB', 'ExtraTrees', 'Ridge+spline', 'SegLGBM']


## M5a — 벤치 실행 · 오차상관 · 선정 · 트랙 valid/test

In [17]:
# ── [Cell 14] M5a — 벤치 실행 + 오차상관 행렬 + 상위2 선정 + 트랙 valid/test 최초기록 ──
import json
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
TOP_TRACK_MODELS = 3          # D8: 트랙(F-T15·F-P3)은 F-C10 상위 3모델에서만 평가

records = []; oof_store = {}
# (A) F-C10 — 후보 전부 필수
print("── (A) F-C10 벤치 (후보 전부) ──")
c10 = {}
for name, fp in CANDS.items():
    r = run_candidate(fp, F_C10); c10[name] = r; oof_store[(name,'F-C10')] = r['oof']
    records.append(dict(model=name, feature_set='F-C10', cv=r['cv'], std=r['std'], time=r['time']))
    print(f"   {name:12s}  CV={r['cv']:7.3f}  std={r['std']:5.3f}  {r['time']:6.1f}s")
# 내장 정합 체크: LGBM×F-C10 == cv_m0b (동일 fold·파라미터)
d0 = abs(c10['LGBM']['cv'] - cv_m0b)
print(f"   [정합] LGBM×F-C10 {c10['LGBM']['cv']:.3f} vs cv_m0b {cv_m0b:.3f}  Δ={d0:.3f}  "
      + ("✅ 재현" if d0 < 0.05 else "⚠️ 불일치 — fold/파라미터 점검"))

order = sorted(c10, key=lambda k: c10[k]['cv'])   # F-C10 CV 오름차순
top3 = order[:TOP_TRACK_MODELS]
print("   F-C10 랭킹:", "  ".join(f"{k}={c10[k]['cv']:.2f}" for k in order))

# (B) 관찰 트랙 — 상위 3모델만 (§8.5 보고 전용, 선정 미개입)
print("── (B) 관찰 트랙 F-T15·F-P3 (상위 3모델) ──")
for name in top3:
    for tname in ['F-T15','F-P3']:
        r = run_candidate(CANDS[name], TRACKS[tname]); oof_store[(name,tname)] = r['oof']
        records.append(dict(model=name, feature_set=tname, cv=r['cv'], std=r['std'], time=r['time']))
        print(f"   {name:12s} {tname}  CV={r['cv']:7.3f}  std={r['std']:5.3f}")

# (C) 벤치 표 (행=모델, 열=피처셋 트랙)
bench = pd.DataFrame(records)
piv = bench.pivot(index='model', columns='feature_set', values='cv').reindex(order)
print("\n[벤치 표] CV\n", piv.round(3).to_string())

# (D) OOF 오차 상관 행렬 (트랙 조합 포함) → M6 앙상블 후보 (§8.3-4)
err = {f"{m}|{t}": oof_store[(m,t)] - y for (m,t) in oof_store}
corr = pd.DataFrame(err).corr()
corr.to_csv(OUT/"model_bench_corr.csv", encoding='utf-8-sig')
lo = corr.where(~np.eye(len(corr), dtype=bool)).stack().idxmin()
print(f"\n[오차 상관] {corr.shape[0]}×{corr.shape[1]} 저장 · 최저 상관쌍 {lo} = {corr.loc[lo]:.3f} (M6 후보)")

# (E) 선정: F-C10 열 CV 최소, |Δ|<0.3 동률이면 빠른 쪽 → 상위 2개 M5b 진출
best_name = order[0]; best_cv = c10[best_name]['cv']
ties = [k for k in order if c10[k]['cv'] - best_cv < 0.3]
sel  = sorted(ties, key=lambda k: c10[k]['time'])[0] if len(ties) > 1 else best_name
m5b  = order[:2]
print(f"\n[선정] F-C10 최소 CV = {best_name} ({best_cv:.3f}); 동률권 {ties} → 채택 {sel}")
print(f"[M5b 진출] 상위 2개: {m5b}  (valid/test는 확인용만 — R10, 선정 미개입)")

# (F) 트랙 valid/test 최초기록 (LGBM 레퍼런스 = §8.5 동결표 기준; F-P3는 M5a 최초 조회)
#     방법론은 M2와 동일(full_train 705 → answer). F-T15는 41.67/43.25 재현되어야 정합.
print("── (F) 트랙 valid/test (LGBM-ref, 마일스톤 1회) ──")
track_vt = {'F-C10': (va_m0b, te_m0b)}
for tname, feats in [('F-T15', F_T15), ('F-P3', F_P3)]:
    mdl = full_train(feats)
    va  = rmse_pred(mdl, Xva, feats, ANS/"valid_Y_answer.csv")
    te  = rmse_pred(mdl, Xte, feats, ANS/"test_Y_answer.csv")
    track_vt[tname] = (va, te)
for tn in ['F-C10','F-T15','F-P3']:
    va, te = track_vt[tn]; print(f"   {tn:6s}  valid={va:.3f}  test={te:.3f}")

# (G) 저장 (results.json에 feature_set 필드 포함 — §8.5-6)
bench.to_csv(OUT/"model_bench.csv", index=False, encoding='utf-8-sig')
json.dump({'bench': records,
           'track_valid_test': {k: {'valid': v[0], 'test': v[1]} for k, v in track_vt.items()},
           'selection': {'best_F-C10': best_name, 'best_cv': best_cv, 'adopted': sel},
           'm5b_top2': list(m5b)},
          open(OUT/"results_M5a.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"\n[저장] outputs/model_bench.csv · outputs/model_bench_corr.csv · outputs/results_M5a.json")
print("  → 다음: M5b — 상위 2개만 Optuna(30~50 trials), F-C10만 튜닝(트랙 재튜닝 금지, 확정 파라미터 전이 1회)")

── (A) F-C10 벤치 (후보 전부) ──
   LGBM          CV= 40.347  std=0.944    84.3s
   XGBoost       CV= 41.435  std=1.019    18.5s
   CatBoost      CV= 41.665  std=1.273    59.9s
   HistGB        CV= 50.970  std=0.492     8.1s
   ExtraTrees    CV= 38.001  std=0.503     8.6s
   Ridge+spline  CV= 67.141  std=0.785     0.5s
   SegLGBM       CV= 40.840  std=0.989   139.5s
   [정합] LGBM×F-C10 40.347 vs cv_m0b 40.347  Δ=0.000  ✅ 재현
   F-C10 랭킹: ExtraTrees=38.00  LGBM=40.35  SegLGBM=40.84  XGBoost=41.43  CatBoost=41.66  HistGB=50.97  Ridge+spline=67.14
── (B) 관찰 트랙 F-T15·F-P3 (상위 3모델) ──
   ExtraTrees   F-T15  CV= 40.906  std=0.993
   ExtraTrees   F-P3  CV= 38.807  std=1.052
   LGBM         F-T15  CV= 43.937  std=1.019
   LGBM         F-P3  CV= 41.052  std=0.969
   SegLGBM      F-T15  CV= 44.274  std=0.862
   SegLGBM      F-P3  CV= 41.492  std=1.045

[벤치 표] CV
 feature_set    F-C10    F-P3   F-T15
model                               
ExtraTrees    38.001  38.807  40.906
LGBM          40.347  41.052  4

In [18]:
add_model = ['XGBoost', 'CatBoost']  #에 대해서 추가로 
for name in add_model:
    for tname in ['F-T15','F-P3']:
        r = run_candidate(CANDS[name], TRACKS[tname]); oof_store[(name,tname)] = r['oof']
        records.append(dict(model=name, feature_set=tname, cv=r['cv'], std=r['std'], time=r['time']))
        print(f"   {name:12s} {tname}  CV={r['cv']:7.3f}  std={r['std']:5.3f}")

   XGBoost      F-T15  CV= 44.816  std=0.890
   XGBoost      F-P3  CV= 42.249  std=1.126
   CatBoost     F-T15  CV= 45.111  std=1.112
   CatBoost     F-P3  CV= 42.254  std=1.327


## ExtraTrees 일반화 점검 (M5b 진입 게이트)
 
> M5a에서 **ExtraTrees(F-C10 CV 38.00)**가 챔피언 LGBM(40.35)을 −2.35pt 앞섰지만, 이는 lot-mate 누수에 취약한 랜덤 wafer-CV 결과입니다.
> 이 셀은 **ExtraTrees의 valid/test를 1회 조회**해 그 우위가 *일반화*인지 *랜덤 CV 암기*인지 판정합니다 (계획 §8.3-3 "valid는 선정 후 확인용").

In [19]:
# ── [Cell 15] M5a 확인 — ExtraTrees vs LGBM valid/test (M5b 진입 게이트) ──
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer

def et_valid_test(feats):
    """ExtraTrees 전체학습 → valid/test RMSE (중앙값 대치는 train에서만 fit → 누수 없음)."""
    imp = SimpleImputer(strategy='median').fit(Xtr[feats])
    m = ExtraTreesRegressor(n_estimators=500, n_jobs=-1, random_state=42).fit(imp.transform(Xtr[feats]), y)
    def _rmse(res, ans_path):
        ans = pd.read_csv(ans_path); ans[ID_COL] = ans[ID_COL].astype(str)
        r = res.copy(); r[ID_COL] = r[ID_COL].astype(str)
        p = m.predict(imp.transform(r[feats]))
        mm = r[[ID_COL]].assign(p=p).merge(ans.rename(columns={TARGET_COL:'t'}), on=ID_COL)
        return float(np.sqrt(np.mean((mm['t']-mm['p'])**2)))
    return _rmse(Xva, ANS/"valid_Y_answer.csv"), _rmse(Xte, ANS/"test_Y_answer.csv")

ET_CV, LG_CV = 38.001, 40.347                 # M5a F-C10 기록값
et_va, et_te = et_valid_test(core10)
print("── M5a 선정 후 확인: ExtraTrees vs LGBM (F-C10) ──")
print(f"   {'모델':<10s} {'CV':>7s} {'valid':>7s} {'test':>7s}   valid−CV")
print(f"   {'LGBM':<10s} {LG_CV:7.3f} {va_m0b:7.3f} {te_m0b:7.3f}   {va_m0b-LG_CV:+.2f}")
print(f"   {'ExtraTrees':<10s} {ET_CV:7.3f} {et_va:7.3f} {et_te:7.3f}   {et_va-ET_CV:+.2f}")

gap = et_va - ET_CV
print(f"\n[판정] ExtraTrees valid−CV = {gap:+.2f}pt  (LGBM은 {va_m0b-LG_CV:+.2f} — 랜덤 CV가 비관적)")
if et_va <= va_m0b - 0.3 and gap < 1.5:
    print("   ✅ 일반화 우위 — ExtraTrees 실재 도전자. M5b에서 ExtraTrees·LGBM 둘 다 튜닝.")
elif et_va >= va_m0b + 0.3:
    print("   ⚠️ 랜덤 CV 과적합(lot-mate 암기 의심) — 챔피언 LGBM 유지, ExtraTrees는 M6 앙상블 다양성 원료로만.")
else:
    print("   ⚖️ 접전 — 잠정 LGBM 유지, M7이 최종 판정.")
print("   ※ 최종 확정은 M7 — GroupKFold(C20) Lot-CV · time-split, 3트랙 분리 보고.")

── M5a 선정 후 확인: ExtraTrees vs LGBM (F-C10) ──
   모델              CV   valid    test   valid−CV
   LGBM        40.347  38.399  38.912   -1.95
   ExtraTrees  38.001  37.124  36.890   -0.88

[판정] ExtraTrees valid−CV = -0.88pt  (LGBM은 -1.95 — 랜덤 CV가 비관적)
   ✅ 일반화 우위 — ExtraTrees 실재 도전자. M5b에서 ExtraTrees·LGBM 둘 다 튜닝.
   ※ 최종 확정은 M7 — GroupKFold(C20) Lot-CV · time-split, 3트랙 분리 보고.


# M5b — 상위 2개(ExtraTrees·LGBM) Optuna 경량 튜닝

## optuna 설치 확인 — 최초 1회

In [20]:
# ── [Cell 15.5] M5b 환경: optuna 설치 확인 ──
import importlib
try:
    import optuna; print(f"✅ optuna {optuna.__version__}")
except ImportError:
    print("⚠️ optuna 미설치 → 아래를 1회 실행:  %pip install optuna")

✅ optuna 4.9.0


 ## Optuna 튜닝 (LGBM·ExtraTrees, F-C10, 랜덤 wafer-CV)

In [21]:
# ── [Cell 16] M5b — 상위 2개 Optuna 경량 튜닝 (F-C10, wafer 5-fold CV 최소화) ──
import optuna
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30                       # §8.3 경량 30~50. LGBM이 느림(≈1trial×5fold) — 30 권장, 시간 여유 시 50.
ET_CV_M5A, LG_CV_M5A = 38.001, 40.347   # M5a F-C10 기본 성능

# --- LGBM 목적함수 (§8.1 탐색공간, 스킴은 M0~M5a와 동일 wafer_cv_oof) ---
def obj_lgbm(trial):
    p = dict(objective='regression', metric='rmse', verbose=-1, seed=42,
        learning_rate    = trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
        num_leaves       = trial.suggest_int('num_leaves', 15, 255),
        min_child_samples= trial.suggest_int('min_child_samples', 5, 100),
        feature_fraction = trial.suggest_float('feature_fraction', 0.5, 1.0),
        bagging_fraction = trial.suggest_float('bagging_fraction', 0.5, 1.0),
        bagging_freq     = trial.suggest_int('bagging_freq', 1, 10),
        lambda_l1        = trial.suggest_float('lambda_l1', 1e-3, 10, log=True),
        lambda_l2        = trial.suggest_float('lambda_l2', 1e-3, 10, log=True),
        min_split_gain   = trial.suggest_float('min_split_gain', 0.0, 1.0))
    return wafer_cv_oof(core10, params=p)

# --- ExtraTrees 목적함수 (§8.3 후보5: n_estimators·max_features·min_samples_leaf + max_depth) ---
def _et_fp(ne, mf, ml, md):                    # 주어진 하이퍼파라미터의 fold fit_pred (중앙값 대치 in-fold)
    def fp(Xa, ya, Xb, yb):
        imp = SimpleImputer(strategy='median').fit(Xa)
        m = ExtraTreesRegressor(n_estimators=ne, max_features=mf, min_samples_leaf=ml,
                                max_depth=md, n_jobs=-1, random_state=42)
        m.fit(imp.transform(Xa), ya); return m.predict(imp.transform(Xb))
    return fp
def obj_et(trial):
    ne = trial.suggest_int('n_estimators', 300, 1200, step=100)
    mf = trial.suggest_categorical('max_features', ['sqrt', 0.5, 0.7, 1.0])
    ml = trial.suggest_int('min_samples_leaf', 1, 20)     # ml>1 = 암기 억제(lot-mate 프로브)
    md = trial.suggest_categorical('max_depth', [None, 12, 20, 32])
    return run_candidate(_et_fp(ne, mf, ml, md), core10)['cv']

st_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
st_lgb.optimize(obj_lgbm, n_trials=N_TRIALS, show_progress_bar=True)
st_et  = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
st_et.optimize(obj_et,  n_trials=N_TRIALS, show_progress_bar=True)

print(f"\n[M5b 튜닝 결과 · F-C10 CV]")
print(f"   LGBM        기본 {LG_CV_M5A:.3f} → 튜닝 {st_lgb.best_value:.3f}  (Δ {st_lgb.best_value-LG_CV_M5A:+.3f})")
print(f"   ExtraTrees  기본 {ET_CV_M5A:.3f} → 튜닝 {st_et.best_value:.3f}  (Δ {st_et.best_value-ET_CV_M5A:+.3f})")
print(f"   LGBM best:  {st_lgb.best_params}")
print(f"   ET   best:  {st_et.best_params}")

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


[M5b 튜닝 결과 · F-C10 CV]
   LGBM        기본 40.347 → 튜닝 40.575  (Δ +0.228)
   ExtraTrees  기본 38.001 → 튜닝 36.929  (Δ -1.072)
   LGBM best:  {'learning_rate': 0.053085958319815364, 'num_leaves': 192, 'min_child_samples': 5, 'feature_fraction': 0.6941353372836316, 'bagging_fraction': 0.9726923727970656, 'bagging_freq': 10, 'lambda_l1': 0.008985046002653421, 'lambda_l2': 0.001153563558256608, 'min_split_gain': 0.3848265007248448}
   ET   best:  {'n_estimators': 600, 'max_features': 1.0, 'min_samples_leaf': 2, 'max_depth': 32}


### (탐색 전용) 관찰 트랙 F-T15·F-P3 Optuna 튜닝

In [ ]:
# ── [Cell 18] (탐색 전용) 트랙 F-T15·F-P3 Optuna 튜닝 — §8.5 예외, 선정·제출 미개입 ──
import json
N_TRIALS_TRK = 30          # 트랙당 시도 수 (F-C10과 동일 예산)
SHOW_VT = False            # True면 트랙 튜닝-베스트의 valid/test도 조회(R10 유의 — 가장 과적합 취약한 수치)

print("="*64)
print("  ⚠️ 탐색 전용 — 트랙 재튜닝(§8.5 예외). champ·제출 불변, 별도 파일 저장.")
print("="*64)

def tune_featureset(feats, n_trials=N_TRIALS_TRK):
    """주어진 피처셋에서 LGBM·ExtraTrees를 각각 Optuna로 튜닝 → 최소 CV·베스트 파라미터."""
    def _obj_lgbm(trial):
        p = dict(objective='regression', metric='rmse', verbose=-1, seed=42,
            learning_rate    = trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            num_leaves       = trial.suggest_int('num_leaves', 15, 255),
            min_child_samples= trial.suggest_int('min_child_samples', 5, 100),
            feature_fraction = trial.suggest_float('feature_fraction', 0.5, 1.0),
            bagging_fraction = trial.suggest_float('bagging_fraction', 0.5, 1.0),
            bagging_freq     = trial.suggest_int('bagging_freq', 1, 10),
            lambda_l1        = trial.suggest_float('lambda_l1', 1e-3, 10, log=True),
            lambda_l2        = trial.suggest_float('lambda_l2', 1e-3, 10, log=True),
            min_split_gain   = trial.suggest_float('min_split_gain', 0.0, 1.0))
        return wafer_cv_oof(feats, params=p)
    def _obj_et(trial):
        ne = trial.suggest_int('n_estimators', 300, 1200, step=100)
        mf = trial.suggest_categorical('max_features', ['sqrt', 0.5, 0.7, 1.0])
        ml = trial.suggest_int('min_samples_leaf', 1, 20)
        md = trial.suggest_categorical('max_depth', [None, 12, 20, 32])
        return run_candidate(_et_fp(ne, mf, ml, md), feats)['cv']
    sl = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    sl.optimize(_obj_lgbm, n_trials=n_trials, show_progress_bar=True)
    se = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    se.optimize(_obj_et,  n_trials=n_trials, show_progress_bar=True)
    return dict(lgbm=dict(cv=sl.best_value, params=sl.best_params),
                et  =dict(cv=se.best_value, params=se.best_params))

# F-C10은 Cell 16에서 이미 튜닝됨 → 재사용. 트랙 2종만 새로 튜닝.
trk_tuned = {'F-C10': dict(lgbm=dict(cv=st_lgb.best_value, params=st_lgb.best_params),
                           et  =dict(cv=st_et.best_value,  params=st_et.best_params))}
for tn in ['F-T15', 'F-P3']:
    print(f"\n[튜닝] {tn} ({len(TRACKS[tn])}f) …")
    trk_tuned[tn] = tune_featureset(TRACKS[tn])

# (선택) 트랙 튜닝-베스트의 valid/test — R10 유의
if SHOW_VT:
    for tn in ['F-T15', 'F-P3']:
        feats = TRACKS[tn]
        # 두 모델 중 CV 낮은 쪽으로 valid/test 조회
        if trk_tuned[tn]['et']['cv'] <= trk_tuned[tn]['lgbm']['cv']:
            va, te = et_full_vt(trk_tuned[tn]['et']['params'], feats); who='ET'
        else:
            pp = dict(objective='regression', metric='rmse', verbose=-1, seed=42, **trk_tuned[tn]['lgbm']['params'])
            mm = full_train(feats, params=pp, rounds=lgbm_best_rounds(pp, feats))
            va = rmse_pred(mm, Xva, feats, ANS/"valid_Y_answer.csv"); te = rmse_pred(mm, Xte, feats, ANS/"test_Y_answer.csv"); who='LGBM'
        trk_tuned[tn]['best_vt'] = dict(model=who, valid=va, test=te)

# 비교 표
print("\n── 트랙 튜닝 CV 비교 (탐색 전용) ──")
print(f"   {'트랙':7s} {'LGBM튜닝':>9s} {'ET튜닝':>9s}   vs F-C10 튜닝(ET {st_et.best_value:.2f})")
for tn in ['F-C10', 'F-T15', 'F-P3']:
    lg, et = trk_tuned[tn]['lgbm']['cv'], trk_tuned[tn]['et']['cv']
    dmin = min(lg, et) - min(st_lgb.best_value, st_et.best_value)
    tag = '' if tn=='F-C10' else (f'  Δ{dmin:+.3f} ' + ('← F-C10 못넘음(코어10 우위)' if dmin >= -0.3 else '← F-C10 상회(M7서 확인)'))
    print(f"   {tn:7s} {lg:9.3f} {et:9.3f}{tag}")

json.dump(trk_tuned, open(Path('outputs')/"results_M5b_track_tuning.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n[저장] outputs/results_M5b_track_tuning.json  (탐색 전용 — 챔피언·제출 불변)")
print(f"  · 챔피언은 여전히 Cell 17의 '{champ}' (F-C10). 이 셀은 트랙 천장 참고치일 뿐.")

  ⚠️ 탐색 전용 — 트랙 재튜닝(§8.5 예외). champ·제출 불변, 별도 파일 저장.

[튜닝] F-T15 (15f) …


  0%|          | 0/30 [00:00<?, ?it/s]

## 확정 valid/test · 챔피언 선정 · 트랙 전이

In [22]:
# ── [Cell 17] M5b — 튜닝 모델 valid/test 확정 + 챔피언 선정 + 트랙 전이 평가 1회 ──
import json
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
LGB_BEST = dict(objective='regression', metric='rmse', verbose=-1, seed=42, **st_lgb.best_params)
ET_BEST  = dict(st_et.best_params)

def lgbm_best_rounds(params, feats=core10):        # 튜닝 lr에 맞는 full-train 예산(fold best_iter 중앙값)
    its = []
    for a, b in FOLDS:
        d  = lgb.Dataset(Xtr[feats].iloc[a], y[a]); dv = lgb.Dataset(Xtr[feats].iloc[b], y[b], reference=d)
        m  = lgb.train(params, d, num_boost_round=5000, valid_sets=[dv], callbacks=[lgb.early_stopping(100, verbose=False)])
        its.append(m.best_iteration)
    return max(int(np.median(its)), 1)

def et_full_vt(params, feats):                     # ExtraTrees 전체학습 → valid/test (대치는 train fit)
    imp = SimpleImputer(strategy='median').fit(Xtr[feats])
    m = ExtraTreesRegressor(n_jobs=-1, random_state=42, **params).fit(imp.transform(Xtr[feats]), y)
    def _r(res, ans):
        a = pd.read_csv(ans); a[ID_COL] = a[ID_COL].astype(str); r = res.copy(); r[ID_COL] = r[ID_COL].astype(str)
        p = m.predict(imp.transform(r[feats]))
        mm = r[[ID_COL]].assign(p=p).merge(a.rename(columns={TARGET_COL:'t'}), on=ID_COL)
        return float(np.sqrt(np.mean((mm['t']-mm['p'])**2)))
    return _r(Xva, ANS/"valid_Y_answer.csv"), _r(Xte, ANS/"test_Y_answer.csv")

# F-C10 튜닝 확정 성능
lg_rnd = lgbm_best_rounds(LGB_BEST)
mlg = full_train(core10, params=LGB_BEST, rounds=lg_rnd)
lg_va = rmse_pred(mlg, Xva, core10, ANS/"valid_Y_answer.csv"); lg_te = rmse_pred(mlg, Xte, core10, ANS/"test_Y_answer.csv")
et_va, et_te = et_full_vt(ET_BEST, core10)

print("── M5b 튜닝 확정 (F-C10) ──")
print(f"   {'모델':<10s} {'CV':>7s} {'valid':>7s} {'test':>7s}")
print(f"   {'LGBM':<10s} {st_lgb.best_value:7.3f} {lg_va:7.3f} {lg_te:7.3f}   (rounds={lg_rnd})")
print(f"   {'ExtraTrees':<10s} {st_et.best_value:7.3f} {et_va:7.3f} {et_te:7.3f}")

# 챔피언 = 튜닝 CV 최소 (동률 |Δ|<0.3이면 단순·빠른 ExtraTrees)
champ = 'ExtraTrees' if st_et.best_value <= st_lgb.best_value + 0.3 else 'LGBM'
best_cv = min(st_et.best_value, st_lgb.best_value)
print(f"\n[챔피언] 튜닝 CV 최소 → {champ}  (M5a 기본 대비 {best_cv - min(ET_CV_M5A, LG_CV_M5A):+.3f})")

# 트랙 전이 평가 (챔피언 확정 파라미터, 재튜닝 없이 1회) — §8.5 rule 2
print("── 트랙 전이 (챔피언 파라미터, 재튜닝 없음) ──")
track_rows = []
for tn in ['F-C10','F-T15','F-P3']:
    feats = TRACKS[tn]
    if champ == 'ExtraTrees':
        cv = st_et.best_value if tn=='F-C10' else run_candidate(
            _et_fp(ET_BEST['n_estimators'], ET_BEST['max_features'], ET_BEST['min_samples_leaf'], ET_BEST['max_depth']), feats)['cv']
        va, te = (et_va, et_te) if tn=='F-C10' else et_full_vt(ET_BEST, feats)
    else:
        cv = st_lgb.best_value if tn=='F-C10' else wafer_cv_oof(feats, params=LGB_BEST)
        if tn=='F-C10': va, te = lg_va, lg_te
        else:
            mm = full_train(feats, params=LGB_BEST, rounds=lgbm_best_rounds(LGB_BEST, feats))
            va = rmse_pred(mm, Xva, feats, ANS/"valid_Y_answer.csv"); te = rmse_pred(mm, Xte, feats, ANS/"test_Y_answer.csv")
    track_rows.append(dict(feature_set=tn, cv=cv, valid=va, test=te))
    print(f"   {tn:6s}  CV {cv:7.3f}  valid {va:7.3f}  test {te:7.3f}")

json.dump({'champion': champ, 'best_cv': best_cv,
           'lgbm': {'params': st_lgb.best_params, 'cv': st_lgb.best_value, 'valid': lg_va, 'test': lg_te, 'rounds': lg_rnd},
           'extratrees': {'params': st_et.best_params, 'cv': st_et.best_value, 'valid': et_va, 'test': et_te},
           'track_transfer': track_rows},
          open(OUT/"results_M5b.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"\n[저장] outputs/results_M5b.json")
print(f"  → 챔피언 {champ}. 단 랜덤 wafer-CV 기준 — **lot-mate 최종 판정은 M7**(GroupKFold C20 Lot-CV·time-split, 3트랙 분리 보고, G3).")

── M5b 튜닝 확정 (F-C10) ──
   모델              CV   valid    test
   LGBM        40.575  38.634  39.325   (rounds=251)
   ExtraTrees  36.929  35.658  35.637

[챔피언] 튜닝 CV 최소 → ExtraTrees  (M5a 기본 대비 -1.072)
── 트랙 전이 (챔피언 파라미터, 재튜닝 없음) ──
   F-C10   CV  36.929  valid  35.658  test  35.637
   F-T15   CV  40.986  valid  38.751  test  39.629
   F-P3    CV  38.491  valid  36.964  test  37.121

[저장] outputs/results_M5b.json
  → 챔피언 ExtraTrees. 단 랜덤 wafer-CV 기준 — **lot-mate 최종 판정은 M7**(GroupKFold C20 Lot-CV·time-split, 3트랙 분리 보고, G3).
